# 16 — Final Results Summary

Aggregates results across every experiment in this thesis. All numbers are loaded directly from `artifacts/*/results.json` written by the source notebooks — no hand-copied figures.

## Structure
- **Phase 1 (NB 02–07):** origin classification — comparing text representation methods (country-level, 14 classes)
- **Scrubbing ablation (NB 04.3–04.5):** progressive data-leakage scrubbing on a fixed model
- **Phase 2 (NB 08–15):** refining the winning paradigm (fine-tuned transformers) — time window, label granularity, pipeline architecture, input modality, training strategy, backbone size
- **Recommendation (NB 05, 05.1):** content-based retrieval via embedding similarity

If a notebook hasn't been re-executed and its JSON is missing, the corresponding row is skipped with a warning.

In [1]:
import json
from pathlib import Path
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

ART = Path('artifacts')

def load(rel_path):
    p = ART / rel_path
    if not p.exists():
        print(f'[skip] missing: {p}')
        return None
    with open(p) as f:
        return json.load(f)

## Phase 1 — Method comparison

Origin classification at the country level (14 classes after the MIN=60 floor) across five representation paradigms. For TF-IDF and Word2Vec, the table reports the best-performing variant from each notebook's grid. For fine-tuned transformers and the LLM, the headline configuration is reported.

In [2]:
rows = []

# 02 TF-IDF — winner among grid variants
d = load('origin_tfidf_fulltext_scrubbed_plus/results.json')
if d:
    w = next(m for m in d['all_models'] if m['model'] == d['winner'])
    rows.append(('02', 'TF-IDF (best variant)', w['model'], w['test_accuracy'], w['test_f1_macro']))

# 03 Word2Vec — winner among grid variants
d = load('origin_word2vec_fulltext_scrubbed_plus/results.json')
if d:
    w = next(m for m in d['all_models'] if m['model'] == d['winner'])
    rows.append(('03', 'Word2Vec / fastText (best variant)', w['model'], w['test_accuracy'], w['test_f1_macro']))

# 04.1 RoBERTa fine-tuned (single-seed headline)
d = load('origin_finetuning_roberta_fulltext_scrubbed_plus/results.json')
if d:
    h = d['headline']
    rows.append(('04.1', 'RoBERTa-base fine-tuned (single seed)', d['model'], h['test_accuracy'], h['test_f1_macro']))

# 04.2 ModernBERT fine-tuned (single-seed headline)
d = load('origin_finetuning_modernbert_fulltext_scrubbed_plus/results.json')
if d:
    h = d['headline']
    rows.append(('04.2', 'ModernBERT-base fine-tuned (single seed)', d['model'], h['test_accuracy'], h['test_f1_macro']))

# 06 LLM zero-shot
d = load('llm_zero_shot_fulltext_scrubbed_plus/results.json')
if d:
    rows.append(('06', 'LLM zero-shot prompting', d['model'], d['accuracy'], d['macro_f1']))

# 07 Seed harness — variance check across 3 seeds
d = load('seed_harness_fulltext_scrubbed_plus/results.json')
if d:
    rh = d['roberta_seed_harness']
    rows.append(('07', 'RoBERTa-base x 3 seeds (mean)', rh['model'], rh['mean']['test_accuracy'], rh['mean']['test_f1_macro']))
    mh = d['modernbert_seed_harness']
    rows.append(('07', 'ModernBERT-base x 3 seeds (mean)', mh['model'], mh['mean']['test_accuracy'], mh['mean']['test_f1_macro']))

phase1 = pd.DataFrame(rows, columns=['NB', 'Setup', 'Model', 'Test acc', 'Test F1'])
phase1

,NB,Setup,Model,Test acc,Test F1
0,02,TF-IDF (best variant),"Linear SVM (word 1-2gram, C=5)",0.7458,0.6264
1,03,Word2Vec / fastText (best variant),fastText + Linear SVM (C=1),0.4956,0.3595
2,04.1,RoBERTa-base fine-tuned (single seed),roberta-base,0.7312,0.6450
3,04.2,ModernBERT-base fine-tuned (single seed),answerdotai/ModernBERT-base,0.7390,0.6113
4,06,LLM zero-shot prompting,Qwen/Qwen2.5-7B-Instruct,0.2522,0.1703
5,07,RoBERTa-base x 3 seeds (mean),roberta-base,0.7335,0.6427
6,07,ModernBERT-base x 3 seeds (mean),answerdotai/ModernBERT-base,0.7230,0.5903


**Phase 1 takeaway:** fine-tuned transformers (RoBERTa-base, ModernBERT-base) clearly dominate classical (TF-IDF) and static-embedding (Word2Vec) baselines on country-level classification. The LLM zero-shot result provides a no-training reference. The seed-harness rows confirm the headline transformer numbers are stable across random seeds.

## Scrubbing ablation (NB 04.3 → 04.4 → 04.5)

Same model (ModernBERT-base) and identical training protocol, varying only how aggressively origin-revealing terms are removed from the input text. The leakage rate is the fraction of test examples that still contain at least one origin-revealing token after scrubbing.

In [3]:
rows = []
for tag, label in [
    ('unscrubbed', '04.3 - No scrubbing (raw)'),
    ('scrubbed', '04.4 - Basic scrubbing (countries only)'),
    ('scrubbed_plus', '04.5 - Scrubbed+ (countries + regions + cultivars + producer)'),
]:
    d = load(f'origin_finetuning_modernbert_fulltext_{tag}/results.json')
    if d:
        h = d['headline']
        leak = d.get('post_scrub_leakage_rate', d.get('leakage_rate'))
        rows.append((label, leak, h['test_accuracy'], h['test_f1_macro']))

ablation = pd.DataFrame(rows, columns=['Tier', 'Leakage rate', 'Test acc', 'Test F1'])
ablation

,Tier,Leakage rate,Test acc,Test F1
0,04.3 - No scrubbing (raw),0.7365,0.9110,0.8881
1,04.4 - Basic scrubbing (countries only),0.0076,0.7556,0.6691
2,04.5 - Scrubbed+ (countries + regions + cultivars + producer),0.0076,0.7390,0.6113


**Ablation takeaway:** unscrubbed performance is inflated by leakage — the model is rewarded for memorizing country names rather than learning flavor signals. Each scrubbing tier reduces leakage and yields a more honest estimate of how well the model can predict origin from *flavor description alone*. The scrubbed+ tier (403 terms across 4 categories) is what every Phase 2 experiment uses.

## Phase 2 — Refinement of the fine-tuned transformer paradigm

After Phase 1 established that fine-tuned transformers are the best paradigm, Phase 2 explores variations *within* that paradigm:

| Axis | Notebooks |
|---|---|
| Time window | NB 08 (10-yr) vs NB 09 (5-yr) |
| Label granularity | country (NB 08, 09) vs region 4-way (NB 10) vs region 5-way (NB 11) |
| Pipeline architecture | flat (NB 10) vs hierarchical (NB 12) |
| Input modality | text-only (NB 10) vs text + numeric scores (NB 13) |
| Training strategy | single-seed (NB 10) vs 3-seed ensemble (NB 14) |
| Backbone size | RoBERTa-base (NB 10) vs RoBERTa-large (NB 15) |

In [4]:
rows = []

# 08 Recent10yr country-level
d = load('origin_recent10yr_min60_scrubbed_plus/results.json')
if d:
    rh = d['roberta_seed_harness']['mean']
    rows.append(('08', '10-yr / MIN=60 / country (14)', 'RoBERTa-base x 3 (mean)', rh['test_accuracy'], rh['test_f1_macro']))

# 09 Recent5yr country-level
d = load('origin_recent5yr_min40_scrubbed_plus/results.json')
if d:
    rh = d['roberta_seed_harness']['mean']
    rows.append(('09', '5-yr / MIN=40 / country (14)', 'RoBERTa-base x 3 (mean)', rh['test_accuracy'], rh['test_f1_macro']))

# 10 Regional 4-way
d = load('origin_region_4way_scrubbed_plus/results.json')
if d:
    rh = d['roberta_seed_harness']['mean']
    rows.append(('10', 'Regional 4-way (E.Africa / C.America / S.America / Asia-Pac)', 'RoBERTa-base x 3 (mean)', rh['test_accuracy'], rh['test_f1_macro']))

# 11 Regional 5-way
d = load('origin_region_5way_scrubbed_plus/results.json')
if d:
    rh = d['roberta_seed_harness']['mean']
    rows.append(('11', 'Regional 5-way (USA split out)', 'RoBERTa-base x 3 (mean)', rh['test_accuracy'], rh['test_f1_macro']))

# 12 Hierarchical region->country (end-to-end, no flat-classifier accuracy reported)
d = load('origin_hierarchical_region_to_country_scrubbed_plus/results.json')
if d:
    s = d['summary']
    e2e_acc_mean = sum(r['e2e_acc'] for r in d['per_seed_eval']) / len(d['per_seed_eval'])
    rows.append(('12', 'Hierarchical region->country (end-to-end)', 'Two-stage RoBERTa', e2e_acc_mean, s['e2e_country_f1_mean']))

# 13 Text + scores fusion
d = load('origin_region_4way_text_plus_scores/results.json')
if d:
    fs = d['fusion_summary']
    rows.append(('13', 'Text + numeric scores fusion (regional 4-way)', 'RoBERTa + MLP x 3 (mean)', fs['test_accuracy_mean'], fs['test_f1_macro_mean']))

# 14 Ensemble of NB 10 seeds
d = load('origin_region_4way_ensemble/results.json')
if d:
    e = d['ensemble']
    rows.append(('14', '3-seed soft-vote ensemble of NB 10 (regional 4-way)', 'RoBERTa-base x 3 ensemble', e['test_accuracy'], e['test_f1_macro']))

# 15 RoBERTa-large
d = load('origin_region_4way_roberta_large/results.json')
if d:
    m = d['mean']
    rows.append(('15', 'RoBERTa-large (regional 4-way)', 'RoBERTa-large x 3 (mean)', m['test_accuracy'], m['test_f1_macro']))

phase2 = pd.DataFrame(rows, columns=['NB', 'Setup', 'Model', 'Test acc', 'Test F1'])
phase2

,NB,Setup,Model,Test acc,Test F1
0,08,10-yr / MIN=60 / country (14),RoBERTa-base x 3 (mean),0.7466,0.6429
1,09,5-yr / MIN=40 / country (14),RoBERTa-base x 3 (mean),0.7094,0.6342
2,10,Regional 4-way (E.Africa / C.America / S.America / Asia-Pac),RoBERTa-base x 3 (mean),0.8345,0.8171
3,11,Regional 5-way (USA split out),RoBERTa-base x 3 (mean),0.8143,0.7999
4,12,Hierarchical region->country (end-to-end),Two-stage RoBERTa,0.6528,0.4829
5,13,Text + numeric scores fusion (regional 4-way),RoBERTa + MLP x 3 (mean),0.8275,0.8082
6,14,3-seed soft-vote ensemble of NB 10 (regional 4-way),RoBERTa-base x 3 ensemble,0.8497,0.8344
7,15,RoBERTa-large (regional 4-way),RoBERTa-large x 3 (mean),0.8477,0.8325


**Phase 2 takeaways:**

- **Region (4-way) >> Country (14-way)** for accuracy: collapsing 14 countries into 4 regions lifts accuracy from ~0.75 to ~0.83.
- **3-seed ensemble (NB 14) is the overall best:** test accuracy **0.8497**, essentially at the 0.85 thesis target.
- **RoBERTa-large matches the ensemble** at 0.8477 with a single model — but at ~3x the inference cost.
- **Negative results worth reporting:**
  - 5-yr time window (NB 09) underperforms 10-yr (less training data).
  - 5-way regional (NB 11) loses ~2 points vs 4-way because USA reviews look like Asia-Pacific.
  - Hierarchical pipeline (NB 12) collapses to F1 ~0.48 vs flat 0.64 — cascade errors compound and per-region heads underperform their flat-classifier slices.
  - Multi-modal fusion (NB 13) costs ~0.7 acc points vs text-only — numeric scores carry little region signal beyond what text already captures.

## Recommendation system

Content-based retrieval evaluated by Precision@10, Recall@10, nDCG@10, MAP@10. Relevance proxy: a returned coffee is relevant if its country matches the query's country.

In [5]:
d = load('recommendation_bge/recommendation_summary.json')
if d:
    rec = pd.DataFrame(d).T
    rec.columns = ['Precision@10', 'Recall@10', 'nDCG@10', 'MAP@10']
else:
    rec = None
rec

,Precision@10,Recall@10,nDCG@10,MAP@10
TF-IDF (baseline),0.1687,0.0029,0.1760,0.0910
SBERT MiniLM-L6 (baseline),0.1652,0.0027,0.1705,0.0886
BGE-base-en-v1.5,0.1794,0.0030,0.1856,0.1018
E5-base-v2,0.1804,0.0030,0.1866,0.1011


**Recommendation takeaway:** modern instruction-tuned retrieval encoders (BGE-base-en-v1.5, E5-base-v2) outperform both classical (TF-IDF) and older dense (SBERT MiniLM-L6) baselines on every metric. The lift is real but modest — roughly 6–12% relative on nDCG / MAP — suggesting the corpus's vocabulary is regular enough that lexical overlap already captures most of the signal, with semantic retrieval providing the remaining headroom.

## Headline numbers (for the thesis abstract)

| Task | Best result | Source |
|---|---|---|
| Country-level origin classification (14 classes) | acc ~0.747, F1 ~0.643 | NB 08 (10-yr / MIN=60, RoBERTa-base x3 mean) |
| **Regional origin classification (4 classes)** | **acc 0.8497, F1 0.8344** | **NB 14 (3-seed soft-vote ensemble)** |
| Recommendation (Precision@10) | 0.180 | NB 05.1 (E5-base-v2) |
| Recommendation (nDCG@10) | 0.187 | NB 05.1 (E5-base-v2) |

## Methodological contributions

1. **Multi-section input** — every origin experiment uses the concatenation of *Blind Assessment + Notes + Who Should Drink It + Bottom Line*, not just the tasting note alone.
2. **4-tier data-leakage scrubbing pipeline** — 403 terms across countries, region aliases, cultivars, and producer-context terms are masked to `[ORIGIN]` before training. The ablation series (NB 04.3 / 04.4 / 04.5) quantifies this.
3. **Regional grouping as a feasibility threshold** — country-level classification plateaus near 0.75 accuracy under honest evaluation; regional grouping (4-way) is the granularity at which the task becomes practically usable.
4. **Seed ensembling as a low-cost lift** — averaging softmax probabilities across 3 seeds of the same architecture adds ~1.5 accuracy points with no additional training cost.